In [0]:
cubeserviceofficetxnservicedate_path = dbutils.widgets.get("cubeserviceofficetxnservicedate_path")
stg_fact_accountreceivablebilled_path = dbutils.widgets.get("stg_fact_accountreceivablebilled_path")
redarfact_path = dbutils.widgets.get("redarfact_path")
office_path = dbutils.widgets.get("office_path")
cubeserviceofficetxnpayor_path = dbutils.widgets.get("cubeserviceofficetxnpayor_path")
client_path=dbutils.widgets.get("client_path")
staging_bkp_accountsreceivablebilled=dbutils.widgets.get("staging_bkp_accountsreceivablebilled")

In [0]:
count = spark.sql(f"SELECT COUNT(*) as cnt FROM {stg_fact_accountreceivablebilled_path}").collect()[0]["cnt"]

if count == 0:
    print("Full load for fact_accountsreceivablebilled executed")

    spark.sql(f"""
        INSERT INTO {stg_fact_accountreceivablebilled_path}
        (
            reportingdate,
            sourcesystemkey,
            officekey,
            payorkey,
            clientkey,
            ar_balance,
            episodestart,
            episodeend,
            firstdos,
            lastdos,
            periodstart,
            periodend,
            days_0_30,
            days_31_60,
            days_61_90,
            days_91_180,
            days_181_270,
            days_270_plus,
            reserverequired,
            agefromtoday,
            agefromquarterend,
            invnum,
            invdate,
            lastbillingnote,
            lastbillingnotetype,
            reportingdatekey
        )
        SELECT
            TRY_CAST(ReportingDate AS DATE) AS reportingdate,
            TRY_CAST(SourceSystemKey AS INT) AS sourcesystemkey,
            TRY_CAST(OfficeKey AS INT) AS officekey,
            TRY_CAST(PayorKey AS INT) AS payorkey,
            TRY_CAST(ClientKey AS INT) AS clientkey,
            TRY_CAST(AR_Balance AS DECIMAL(18,2)) AS ar_balance,

            TRY_CAST(EpisodeStart AS DATE) AS episodestart,
            TRY_CAST(EpisodeEnd AS DATE) AS episodeend,

            TRY_CAST(FirstDOS AS DATE) AS firstdos,
            TRY_CAST(LastDOS AS DATE) AS lastdos,

            TRY_CAST(PeriodStart AS DATE) AS periodstart,
            TRY_CAST(PeriodEnd AS DATE) AS periodend,

            TRY_CAST(`0___30_Days` AS DECIMAL(18,2)) AS days_0_30,
            TRY_CAST(`31___60_Days` AS DECIMAL(18,2)) AS days_31_60,
            TRY_CAST(`61___90_Days` AS DECIMAL(18,2)) AS days_61_90,
            TRY_CAST(`91___180_Days` AS DECIMAL(18,2)) AS days_91_180,
            TRY_CAST(`181___270_Days` AS DECIMAL(18,2)) AS days_181_270,

            -- PRD column name is `271__Days` but DEV expects days_270_plus
            TRY_CAST(`271__Days` AS DECIMAL(18,2)) AS days_270_plus,

            TRY_CAST(ReserveRequired AS DECIMAL(18,2)) AS reserverequired,
            TRY_CAST(AgeFromToday AS INT) AS agefromtoday,
            TRY_CAST(AgeFromQuarterEnd AS INT) AS agefromquarterend,

            TRY_CAST(InvNum AS STRING) AS invnum,
            TRY_CAST(InvDate AS DATE) AS invdate,

            TRY_CAST(LastBillingNote AS STRING) AS lastbillingnote,
            TRY_CAST(LastBillingNoteType AS STRING) AS lastbillingnotetype,

            TRY_CAST(ReportingDateKey AS STRING) AS reportingdatekey

        FROM {staging_bkp_accountsreceivablebilled}
        WHERE SourceSystemKey IN (6)
    """)


In [0]:
spark.sql(
    f"""  
-- Declare and set the last day in quarter variable
CREATE OR REPLACE TEMPORARY VIEW LastDayInQtr AS
SELECT MAX(servicedate) as LastDayInQtr
FROM {cubeserviceofficetxnservicedate_path}
WHERE ServiceQuarterNbr = (
  SELECT ServiceQuarterNbr 
  FROM {cubeserviceofficetxnservicedate_path} 
  WHERE ServiceDate = CURRENT_DATE()
);
    """
)

In [0]:
run_flag = spark.sql("SELECT dayofweek(current_date()) as dow").collect()[0]["dow"]

if run_flag == 1:
  spark.sql(
      f""" 
  INSERT INTO {stg_fact_accountreceivablebilled_path} (
    reportingdate, sourcesystemkey, officekey, payorkey, clientkey, ar_balance,
    episodestart, episodeend, firstdos, lastdos, periodstart, periodend, 
    days_0_30, days_31_60, days_61_90, days_91_180, 
    days_181_270, days_270_plus, reserverequired, agefromtoday, agefromquarterend, 
    invnum, invdate, lastbillingnote, lastbillingnotetype, reportingdatekey
  )
  SELECT 
    CURRENT_DATE() as reportingdate,
    6 as sourcesystemkey,
    ofc.OfficeKey as officekey,
    Py.PayorKey as payorkey,
    csi.ClientKey as clientkey,
    raf.stillowe as ar_balance,
    raf.episodestart,
    raf.episodeend,
    TO_DATE(raf.firstdos, 'MM/dd/yyyy') as firstdos,
    TO_DATE(raf.lastdos, 'MM/dd/yyyy') as lastdos,
    raf.periodstart,
    raf.periodend,
    
    -- 0-30 Days
    CASE 
      WHEN raf.pdgmperiodid IS NOT NULL THEN
        COALESCE(
          CASE WHEN raf.periodend <= CURRENT_DATE() 
              AND raf.periodend >= DATE_ADD(CURRENT_DATE(), -30)
          THEN COALESCE(raf.stillowe, 0) END, 0)
      WHEN raf.payortype LIKE '%PPS%' THEN
        COALESCE(
          CASE WHEN raf.episodeend <= CURRENT_DATE() 
              AND raf.episodeend >= DATE_ADD(CURRENT_DATE(), -30)
          THEN COALESCE(raf.stillowe, 0) END, 0)
      ELSE
        COALESCE(
          CASE WHEN TO_DATE(raf.lastdos, 'MM/dd/yyyy') <= CURRENT_DATE() 
              AND TO_DATE(raf.lastdos, 'MM/dd/yyyy') >= DATE_ADD(CURRENT_DATE(), -30)
          THEN COALESCE(raf.stillowe, 0) END, 0)
    END as days_0_30,
    
    -- 31-60 Days
    CASE 
      WHEN raf.pdgmperiodid IS NOT NULL THEN
        COALESCE(
          CASE WHEN raf.periodend <= DATE_ADD(CURRENT_DATE(), -31)
              AND raf.periodend >= DATE_ADD(CURRENT_DATE(), -61)
          THEN COALESCE(raf.stillowe, 0) END, 0)
      WHEN raf.payortype LIKE '%PPS%' THEN
        COALESCE(
          CASE WHEN raf.episodeend <= DATE_ADD(CURRENT_DATE(), -31)
              AND raf.episodeend >= DATE_ADD(CURRENT_DATE(), -61)
          THEN COALESCE(raf.stillowe, 0) END, 0)
      ELSE
        COALESCE(
          CASE WHEN TO_DATE(raf.lastdos, 'MM/dd/yyyy') <= DATE_ADD(CURRENT_DATE(), -31)
              AND TO_DATE(raf.lastdos, 'MM/dd/yyyy') >= DATE_ADD(CURRENT_DATE(), -61)
          THEN COALESCE(raf.stillowe, 0) END, 0)
    END as days_31_60,
    
    -- 61-90 Days
    CASE 
      WHEN raf.pdgmperiodid IS NOT NULL THEN
        COALESCE(
          CASE WHEN raf.periodend <= DATE_ADD(CURRENT_DATE(), -62)
              AND raf.periodend >= DATE_ADD(CURRENT_DATE(), -92)
          THEN COALESCE(raf.stillowe, 0) END, 0)
      WHEN raf.payortype LIKE '%PPS%' THEN
        COALESCE(
          CASE WHEN raf.episodeend <= DATE_ADD(CURRENT_DATE(), -62)
              AND raf.episodeend >= DATE_ADD(CURRENT_DATE(), -92)
          THEN COALESCE(raf.stillowe, 0) END, 0)
      ELSE
        COALESCE(
          CASE WHEN TO_DATE(raf.lastdos, 'MM/dd/yyyy') <= DATE_ADD(CURRENT_DATE(), -62)
              AND TO_DATE(raf.lastdos, 'MM/dd/yyyy') >= DATE_ADD(CURRENT_DATE(), -92)
          THEN COALESCE(raf.stillowe, 0) END, 0)
    END as days_61_90,
    
    -- 91-180 Days
    CASE 
      WHEN raf.pdgmperiodid IS NOT NULL THEN
        COALESCE(
          CASE WHEN raf.periodend <= DATE_ADD(CURRENT_DATE(), -93)
              AND raf.periodend >= DATE_ADD(CURRENT_DATE(), -181)
          THEN COALESCE(raf.stillowe, 0) END, 0)
      WHEN raf.payortype LIKE '%PPS%' THEN
        COALESCE(
          CASE WHEN raf.episodeend <= DATE_ADD(CURRENT_DATE(), -93)
              AND raf.episodeend >= DATE_ADD(CURRENT_DATE(), -181)
          THEN COALESCE(raf.stillowe, 0) END, 0)
      ELSE
        COALESCE(
          CASE WHEN TO_DATE(raf.lastdos, 'MM/dd/yyyy') <= DATE_ADD(CURRENT_DATE(), -93)
              AND TO_DATE(raf.lastdos, 'MM/dd/yyyy') >= DATE_ADD(CURRENT_DATE(), -181)
          THEN COALESCE(raf.stillowe, 0) END, 0)
    END as days_91_180,
    
    -- 181-270 Days
    CASE 
      WHEN raf.pdgmperiodid IS NOT NULL THEN
        COALESCE(
          CASE WHEN raf.periodend <= DATE_ADD((SELECT LastDayInQtr FROM LastDayInQtr), -182)
              AND raf.periodend >= DATE_ADD((SELECT LastDayInQtr FROM LastDayInQtr), -271)
          THEN COALESCE(raf.stillowe, 0) END, 0)
      WHEN raf.payortype LIKE '%PPS%' THEN
        COALESCE(
          CASE WHEN raf.episodeend <= DATE_ADD((SELECT LastDayInQtr FROM LastDayInQtr), -182)
              AND raf.episodeend >= DATE_ADD((SELECT LastDayInQtr FROM LastDayInQtr), -271)
          THEN COALESCE(raf.stillowe, 0) END, 0)
      ELSE
        COALESCE(
          CASE WHEN TO_DATE(raf.lastdos, 'MM/dd/yyyy') <= DATE_ADD((SELECT LastDayInQtr FROM LastDayInQtr), -182)
              AND TO_DATE(raf.lastdos, 'MM/dd/yyyy') >= DATE_ADD((SELECT LastDayInQtr FROM LastDayInQtr), -271)
          THEN COALESCE(raf.stillowe, 0) END, 0)
    END as days_181_270,
    
    -- 270+ Days
    CASE 
      WHEN raf.pdgmperiodid IS NOT NULL THEN
        COALESCE(
          CASE WHEN raf.periodend <= DATE_ADD((SELECT LastDayInQtr FROM LastDayInQtr), -272)
          THEN COALESCE(raf.stillowe, 0) END, 0)
      WHEN raf.payortype LIKE '%PPS%' THEN
        COALESCE(
          CASE WHEN raf.episodeend <= DATE_ADD((SELECT LastDayInQtr FROM LastDayInQtr), -272)
          THEN COALESCE(raf.stillowe, 0) END, 0)
      ELSE
        COALESCE(
          CASE WHEN TO_DATE(raf.lastdos, 'MM/dd/yyyy') <= DATE_ADD((SELECT LastDayInQtr FROM LastDayInQtr), -272)
          THEN COALESCE(raf.stillowe, 0) END, 0)
    END as days_270_plus,
    
    -- Reserve Required (NULL placeholder - add logic if needed)
    NULL as reserverequired,
    
    -- Age from Today
    CASE 
      WHEN raf.pdgmperiodid IS NOT NULL THEN DATEDIFF(CURRENT_DATE(), raf.periodend)
      WHEN raf.payortype LIKE '%PPS%' THEN DATEDIFF(CURRENT_DATE(), raf.episodeend)
      ELSE DATEDIFF(CURRENT_DATE(), TO_DATE(raf.lastdos, 'MM/dd/yyyy'))
    END as agefromtoday,
    
    -- Age from Quarter End
    CASE 
      WHEN raf.pdgmperiodid IS NOT NULL THEN DATEDIFF((SELECT LastDayInQtr FROM LastDayInQtr), raf.periodend)
      WHEN raf.payortype LIKE '%PPS%' THEN DATEDIFF((SELECT LastDayInQtr FROM LastDayInQtr), raf.episodeend)
      ELSE DATEDIFF((SELECT LastDayInQtr FROM LastDayInQtr), TO_DATE(raf.lastdos, 'MM/dd/yyyy'))
    END as agefromquarterend,
    
    CAST(raf.invnum AS STRING) as invnum,
    TO_DATE(raf.billdate, 'MM/dd/yyyy') as invdate,
    raf.lastbillingnote,
    raf.lastbillingnotetype,
    
    -- Reporting Date Key (format: YYYYMMDD)
    DATE_FORMAT(CURRENT_DATE(), 'yyyyMMdd') as reportingdatekey

  FROM {redarfact_path} raf
  LEFT JOIN {office_path} ofc
    ON ABS(raf.branchid) = ofc.OfficeNumber
  LEFT JOIN {cubeserviceofficetxnpayor_path} py
    ON CAST(py.payorid AS STRING) = CAST(raf.ps_id AS STRING)
  LEFT JOIN (
    SELECT * FROM (
      SELECT 
        sourcesystemID, 
        clientkey, 
        SourceSystem,
        ROW_NUMBER() OVER (PARTITION BY sourcesystemID, rc.SourceSystem ORDER BY sourcesystemID) as rnk1
      FROM {client_path} rc
    ) rc1 
    WHERE rc1.rnk1 = 1
  ) csi
    ON CAST(raf.episodeid AS STRING) = LTRIM(RTRIM(CAST(csi.SourceSystemId AS STRING)))
    AND csi.SourceSystem = 'HCHB'
    WHERE dayofweek(current_date()) = 1;

  """
  )

In [0]:
run_flag = spark.sql("SELECT dayofweek(current_date()) as dow").collect()[0]["dow"]

if run_flag == 1: 
    spark.sql(
        f"""
    UPDATE {stg_fact_accountreceivablebilled_path}
    SET reportingdatekey = REPLACE(CAST(reportingdate AS STRING), '-', '')
    WHERE reportingdate = CURRENT_DATE();
    """
    )

In [0]:
run_flag = spark.sql("SELECT dayofweek(current_date()) as dow").collect()[0]["dow"]

if run_flag == 1:
    spark.sql(
        f""" 
    DELETE FROM {{stg_fact_accountreceivablebilled_path}}
    WHERE (SourceSystemKey, officekey, reportingdate, sourcesystemkey, payorkey, clientkey, ar_balance,
        episodestart, episodeend, firstdos, lastdos, periodstart, periodend,
        days_0_30, days_31_60, days_61_90, days_91_180,
        days_181_270, days_270_plus, reserverequired, agefromtoday, agefromquarterend,
        invnum, invdate, lastbillingnote, lastbillingnotetype, reportingdatekey)
    IN
    (
    SELECT SourceSystemKey, officekey, reportingdate, sourcesystemkey, payorkey, clientkey, ar_balance,
            episodestart, episodeend, firstdos, lastdos, periodstart, periodend,
            days_0_30, days_31_60, days_61_90, days_91_180,
            days_181_270, days_270_plus, reserverequired, agefromtoday, agefromquarterend,
            invnum, invdate, lastbillingnote, lastbillingnotetype, reportingdatekey
    FROM (
        SELECT *,
                ROW_NUMBER() OVER (
                PARTITION BY reportingdate, sourcesystemkey, officekey, payorkey, clientkey, ar_balance,
                                episodestart, episodeend, firstdos, lastdos, periodstart, periodend,
                                days_0_30, days_31_60, days_61_90, days_91_180,
                                days_181_270, days_270_plus, reserverequired, agefromtoday, agefromquarterend,
                                invnum, invdate, lastbillingnote, lastbillingnotetype, reportingdatekey
                ORDER BY invnum
                ) AS rnb
        FROM {stg_fact_accountreceivablebilled_path}
        WHERE SourceSystemKey = 6
            AND officekey IN (496, 1219)
    ) t
    WHERE rnb > 1
    );
    """
    )
